In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/README.md
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18184.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18177.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16773.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19830.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16786.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.hea
/kag

In [2]:
# ============================================================
# Imports
# ============================================================

import os
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Reproducibility
np.random.seed(42)
random.seed(42)

In [3]:
# ============================================================
# Dataset Paths
# ============================================================

# Notebook 03 Output
NOTEBOOK3_ROOT = "/kaggle/input/notebooks/mdrashidshahariar/notebook-03-paper-dataset-construction-2-minute"

# Original ECG Dataset (only for reference)
ECG_ROOT = "/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data"

# Balanced Dataset generated in Notebook 3
BALANCED_ROOT = os.path.join(
    NOTEBOOK3_ROOT,
    "Balanced_Dataset"
)

# Notebook 4 output
OUTPUT_ROOT = "/kaggle/working/Prepared_Dataset"

prediction_windows = [30, 25, 20, 15, 10, 5]

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)

print("Balanced Dataset :", BALANCED_ROOT)
print("Output Folder    :", OUTPUT_ROOT)

Balanced Dataset : /kaggle/input/notebooks/mdrashidshahariar/notebook-03-paper-dataset-construction-2-minute/Balanced_Dataset
Output Folder    : /kaggle/working/Prepared_Dataset


In [4]:
# ============================================================
# Verify Dataset
# ============================================================

for minute in prediction_windows:

    positive = os.path.join(
        BALANCED_ROOT,
        f"{minute}_min",
        "Positive"
    )

    negative = os.path.join(
        BALANCED_ROOT,
        f"{minute}_min",
        "Negative"
    )

    positive_count = len([
        f for f in os.listdir(positive)
        if f.endswith(".npy")
    ])

    negative_count = len([
        f for f in os.listdir(negative)
        if f.endswith(".npy")
    ])

    print("="*40)
    print(f"{minute} Minutes")
    print("="*40)
    print("Positive :", positive_count)
    print("Negative :", negative_count)

30 Minutes
Positive : 1200
Negative : 1200
25 Minutes
Positive : 1200
Negative : 1200
20 Minutes
Positive : 1200
Negative : 1200
15 Minutes
Positive : 1200
Negative : 1200
10 Minutes
Positive : 1200
Negative : 1200
5 Minutes
Positive : 1200
Negative : 1200


In [5]:
# ============================================================
# Verify ECG Segment
# ============================================================

sample_path = os.path.join(
    BALANCED_ROOT,
    "30_min",
    "Positive",
    sorted(
        os.listdir(
            os.path.join(
                BALANCED_ROOT,
                "30_min",
                "Positive"
            )
        )
    )[0]
)

sample = np.load(sample_path)

print("Shape :", sample.shape)
print("Type  :", sample.dtype)
print("Min   :", sample.min())
print("Max   :", sample.max())

Shape : (500, 2)
Type  : float64
Min   : 0.0475
Max   : 0.82125


In [6]:
# ============================================================
# Load Balanced Dataset
# ============================================================

def load_prediction_dataset(minute):

    X = []
    y = []

    positive_folder = os.path.join(
        BALANCED_ROOT,
        f"{minute}_min",
        "Positive"
    )

    negative_folder = os.path.join(
        BALANCED_ROOT,
        f"{minute}_min",
        "Negative"
    )

    # Positive class
    for file in sorted(os.listdir(positive_folder)):

        if file.endswith(".npy"):

            ecg = np.load(
                os.path.join(
                    positive_folder,
                    file
                )
            )

            X.append(ecg)
            y.append(1)

    # Negative class
    for file in sorted(os.listdir(negative_folder)):

        if file.endswith(".npy"):

            ecg = np.load(
                os.path.join(
                    negative_folder,
                    file
                )
            )

            X.append(ecg)
            y.append(0)

    X = np.array(X)
    y = np.array(y)

    return X, y

In [7]:
# ============================================================
# Test Dataset Loader
# ============================================================

X, y = load_prediction_dataset(30)

print("ECG Shape :", X.shape)
print("Labels Shape :", y.shape)

print()

print("Positive :", np.sum(y == 1))
print("Negative :", np.sum(y == 0))

ECG Shape : (2400, 500, 2)
Labels Shape : (2400,)

Positive : 1200
Negative : 1200


In [8]:
# ============================================================
# Load All Prediction Horizon Datasets
# ============================================================

datasets = {}

for minute in prediction_windows:

    X, y = load_prediction_dataset(minute)

    datasets[minute] = {
        "X": X,
        "y": y
    }

    print("=" * 50)
    print(f"{minute} Minutes")
    print("=" * 50)
    print(f"ECG Shape : {X.shape}")
    print(f"Labels    : {y.shape}")
    print(f"Positive  : {np.sum(y == 1)}")
    print(f"Negative  : {np.sum(y == 0)}")

30 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200
25 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200
20 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200
15 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200
10 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200
5 Minutes
ECG Shape : (2400, 500, 2)
Labels    : (2400,)
Positive  : 1200
Negative  : 1200


In [9]:
# ============================================================
# Save Prepared Dataset
# ============================================================

for minute in prediction_windows:

    save_folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min"
    )

    os.makedirs(
        save_folder,
        exist_ok=True
    )

    np.save(
        os.path.join(save_folder, "X.npy"),
        datasets[minute]["X"]
    )

    np.save(
        os.path.join(save_folder, "y.npy"),
        datasets[minute]["y"]
    )

print("Prepared datasets saved successfully.")

Prepared datasets saved successfully.


In [10]:
# ============================================================
# Verify Saved Dataset
# ============================================================

for minute in prediction_windows:

    folder = os.path.join(
        OUTPUT_ROOT,
        f"{minute}_min"
    )

    X = np.load(
        os.path.join(folder, "X.npy")
    )

    y = np.load(
        os.path.join(folder, "y.npy")
    )

    print("=" * 50)
    print(f"{minute} Minutes")
    print("=" * 50)
    print("X Shape :", X.shape)
    print("y Shape :", y.shape)

30 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
25 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
20 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
15 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
10 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
5 Minutes
X Shape : (2400, 500, 2)
y Shape : (2400,)
